# scSketch multi-view integration demo

This notebook is a small smoke test for the integrated `ScSketch(extra_views=...)` API. It is separate from `umap_pca_linked_view_suppfig.ipynb` so the reviewer-facing supplementary-figure workflow stays clean.

## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from jscatter import Scatter

from scsketch import ScSketch

## Configure input

In [2]:
DATA_FILENAME = "trajectory_task_dataset_v1_participant.h5ad"

# Optional: use a smaller subset while testing the UI.
SUBSET_COL = None
SUBSET_VALUES = None

# Optional: set this after inspecting adata.obs columns.
LABEL_COL = None

N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 30
UMAP_RANDOM_STATE = 0

DATA_PATH_CANDIDATES = [
    Path("Manuscript/revision_analyses/data") / DATA_FILENAME,
    Path("data") / DATA_FILENAME,
    Path("outputs/data") / DATA_FILENAME,
    Path("Manuscript/revision_analyses/outputs/data") / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    checked = "\n".join(f"- {path}" for path in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(f"Could not find {DATA_FILENAME}. Checked:\n{checked}")

print(f"Using AnnData: {DATA_PATH}")

Using AnnData: data/trajectory_task_dataset_v1_participant.h5ad


## Load data

In [3]:
adata_raw = sc.read_h5ad(DATA_PATH)
print(adata_raw)
print("obs columns:", list(adata_raw.obs.columns))
print("obsm keys:", list(adata_raw.obsm.keys()))

AnnData object with n_obs × n_vars = 6188 × 20222
    obs: 'cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading'
    var: 'id', 'gene_short_name', 'num_cells_expressed'
    uns: 'download_files', 'source', 'source_url'
    layers: 'counts'
obs columns: ['cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading']
obsm keys: []


In [4]:
adata = adata_raw.copy()

if SUBSET_COL is not None and SUBSET_VALUES is not None:
    if SUBSET_COL not in adata.obs.columns:
        raise KeyError(f"SUBSET_COL={SUBSET_COL!r} is not in adata.obs.")
    keep = adata.obs[SUBSET_COL].astype(str).isin([str(value) for value in SUBSET_VALUES])
    adata = adata[keep].copy()
    print(f"Subset {SUBSET_COL} to {SUBSET_VALUES}: {adata.n_obs:,} cells")
else:
    print(f"Using all cells: {adata.n_obs:,}")

if adata.n_obs < 50:
    raise ValueError("The selected subset is very small. Pick a broader subset for visualization.")

adata

Using all cells: 6,188


AnnData object with n_obs × n_vars = 6188 × 20222
    obs: 'cell', 'n.umi', 'time.point', 'batch', 'Size_Factor', 'raw.embryo.time', 'embryo.time', 'embryo.time.bin', 'raw.embryo.time.bin', 'lineage', 'num_genes_expressed', 'cell.type', 'bg.300.loading', 'bg.400.loading', 'bg.500.1.loading', 'bg.500.2.loading', 'bg.r17.loading', 'bg.b01.loading', 'bg.b02.loading'
    var: 'id', 'gene_short_name', 'num_cells_expressed'
    uns: 'download_files', 'source', 'source_url'
    layers: 'counts'

## Pick metadata for shared coloring

In [5]:
CANDIDATE_METADATA_COLS = [
    "Cell.Type",
    "cell_type",
    "celltype",
    "cell_type_short",
    "cell_state",
    "cell.type",
    "lineage",
    "timepoint",
    "time.point",
    "embryo.time.bin",
    "embryo.time",
    "cluster",
    "clusters",
    "louvain",
    "leiden",
    "seurat_clusters",
    "partition",
    "time",
]

metadata_cols = [col for col in CANDIDATE_METADATA_COLS if col in adata.obs.columns]

if LABEL_COL is None:
    COLOR_BY = metadata_cols[0] if metadata_cols else None
else:
    if LABEL_COL not in adata.obs.columns:
        raise KeyError(f"LABEL_COL={LABEL_COL!r} is not in adata.obs.")
    COLOR_BY = LABEL_COL
    if COLOR_BY not in metadata_cols:
        metadata_cols = [COLOR_BY] + metadata_cols

if COLOR_BY is None:
    adata.obs["all_cells"] = "all_cells"
    metadata_cols = ["all_cells"]
    COLOR_BY = "all_cells"

print("Metadata columns available in scSketch:", metadata_cols)
print("Default color:", COLOR_BY)

Metadata columns available in scSketch: ['cell.type', 'lineage', 'time.point', 'embryo.time.bin', 'embryo.time']
Default color: cell.type


## Ensure UMAP and PCA are available

In [6]:
if "X_umap" not in adata.obsm or "X_pca" not in adata.obsm:
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    if adata.n_vars > N_TOP_GENES:
        sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES)
        adata = adata[:, adata.var["highly_variable"]].copy()
        print(f"Restricted to {adata.n_vars:,} highly variable genes")

    n_comps = min(N_PCS, adata.n_obs - 1, adata.n_vars - 1)
    if n_comps < 2:
        raise ValueError("Need at least two PCA components for multi-view testing.")

    sc.pp.pca(adata, n_comps=n_comps)
    sc.pp.neighbors(adata, n_neighbors=min(N_NEIGHBORS, adata.n_obs - 1), n_pcs=n_comps)
    sc.tl.umap(adata, random_state=UMAP_RANDOM_STATE)
else:
    print("Using existing adata.obsm['X_umap'] and adata.obsm['X_pca']")

print(f"X_umap shape: {adata.obsm['X_umap'].shape}")
print(f"X_pca shape: {adata.obsm['X_pca'].shape}")

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Restricted to 2,000 highly variable genes
X_umap shape: (6188, 2)
X_pca shape: (6188, 50)


## Build the extra PCA scatter view

In [7]:
pca_columns = {
    "obs_name": adata.obs_names.astype(str),
    "PC1": np.asarray(adata.obsm["X_pca"][:, 0], dtype=float),
    "PC2": np.asarray(adata.obsm["X_pca"][:, 1], dtype=float),
}

for col in metadata_cols:
    pca_columns[col] = adata.obs[col].astype(str).to_numpy()

pca_df = pd.DataFrame(pca_columns, index=adata.obs_names)

pca = Scatter(
    data=pca_df,
    x="PC1",
    y="PC2",
    color_by=COLOR_BY,
    width=420,
    height=420,
    axes=True,
    legend=False,
    tooltip=True,
    tooltip_properties=["obs_name", COLOR_BY],
)

## Launch scSketch with the integrated multi-view panel

In [10]:
sketch = ScSketch(
    adata=adata,
    metadata_cols=metadata_cols,
    color_by_default=COLOR_BY,
    height=620,
    max_genes=0,
    extra_views={"PCA": pca},
)

sketch.show()

## Quick checks

- Brush or lasso cells in the main scSketch UMAP view and confirm the same points are selected in the PCA panel.
- Select points in the PCA panel and confirm the main scSketch view updates.
- Change `Color By` in scSketch and confirm the PCA panel uses the same category colors when that metadata column exists in `pca_df`.
- After computing directional results, click a result gene and confirm the UMAP and PCA both use the same gene-expression coloring.
- Toggle **Multi-view** OFF and click a result gene to confirm the normal gene-detail panel returns.